In [1]:
import os
import time
from calendar import timegm
from datetime import date, datetime

import requests
from urllib3 import disable_warnings
from urllib3.exceptions import InsecureRequestWarning

today = date.today()
base = "http://monipe-central.rnp.br"


def get_response(url, time_range):
    cont = 0
    while True:
        header = {"time-range": time_range}
        response = requests.get(url, params=header, verify=False)
        if response.status_code == 200:
            return response
        else:
            cont += 1
            print(f"Received Status Code {response.status_code}. Trying again in 1 min...")
            print(f"Trial: {cont}")
            time.sleep(60)


def get_data(url, time_range):
    response = get_response(url, time_range)
    return response.json()


def request_by_metadata_key(url, type):
    response = requests.get(url, verify=False)
    if response.status_code == 200:
        for obj in response.json()["event-types"]:
            if obj["event-type"] == type:
                return obj
    return response.status_code


def calc_mean(val):
    values = [float(key) for key in val]
    return round(sum(values) / len(values), 2)


def request(folder, name, source, destination, type, time_range, target_bandwidth="9999999999"):
    disable_warnings(InsecureRequestWarning)
    url = "http://monipe-central.rnp.br/esmond/perfsonar/archive/?"
    header = {
        "pscheduler-test-type": type,
        "source": source,
        "destination": destination,
        "bw-target-bandwidth": target_bandwidth,
        "time-range": time_range,
    }
    response = requests.get(url, params=header, verify=False)
    print("endereço", response.url)

    if not os.path.exists(folder):
        os.makedirs(folder)

    if response.status_code == 200:
        datas = []
        for obj in response.json():
            url_obj = obj["url"]
            base_obj = request_by_metadata_key(url_obj, type)
            if not isinstance(base_obj, int):
                url_base = base_obj["base-uri"]
                data = get_data(base + url_base, time_range)
                datas.append(data)

        with open(
            f"{folder}{name} esmond data {source.split('-')[1]}-{destination.split('-')[1]} {today.strftime('%m-%d-%Y')}.csv",
            "w",
        ) as f:
            f.write("Timestamp,Data,Vazao\n")
            for data in datas:
                for obj in data:
                    f.write(
                        f"{int(obj['ts'])},{datetime.fromtimestamp(int(obj['ts'])).strftime('%Y-%m-%d %H:%M:%S')},{str(obj['val'])}\n"
                    )


def request_traceroute(folder, name, source, destination, type, time_range):
    disable_warnings(InsecureRequestWarning)
    limite = "?limit=26400"
    url = "http://monipe-central.rnp.br/esmond/perfsonar/archive/?"
    header = {
        "pscheduler-test-type": type,
        "source": source,
        "destination": destination,
        "time-range": time_range,
    }
    response = requests.get(url, params=header, verify=False)

    if not os.path.exists(folder):
        os.makedirs(folder)

    if response.status_code == 200:
        bases = []
        for obj in response.json():
            for obj_types in obj["event-types"]:
                if obj_types.get("event-type") == "packet-trace":
                    bases.append(obj_types.get("base-uri"))
                    break

        with open(
            f"{folder}{name} esmond data {source.split('-')[1]}-{destination.split('-')[1]} {today.strftime('%m-%d-%Y')}.csv",
            "w",
        ) as f:
            # Escrever cabeçalho, pode adaptar conforme necessidade
            f.write("Timestamp,Datetime,HopHostnames\n")
            for link in bases:
                values = get_data(base + link + limite, time_range)
                for obj in values:
                    ts = int(obj.get("ts", 0))
                    dt = datetime.fromtimestamp(ts).strftime('%Y-%m-%d %H:%M:%S')
                    hop_hostnames = []
                    for hop in obj.get("val", []):
                        try:
                            hop_hostnames.append(hop.get("hostname", "'No Hostname'"))
                        except:
                            hop_hostnames.append("'No Hostname'")
                    # Escreve o timestamp, data e todos os hops concatenados separados por ";"
                    f.write(f"{ts},{dt},{';'.join(hop_hostnames)}\n")


def request_atraso(folder, name, source, destination, type, time_range, label):
    disable_warnings(InsecureRequestWarning)
    limite = "?limit=285000"
    url = "http://monipe-central.rnp.br/esmond/perfsonar/archive/?"
    header = {
        "pscheduler-test-type": type,
        "source": source,
        "destination": destination,
        "time-range": time_range,
    }
    response = requests.get(url, params=header, verify=False)
    print(f"URL: {response.url}")
    print(f"Status Code: {response.status_code}")
    print(f"Resposta: {response.json()}")

    if not os.path.exists(folder):
        os.makedirs(folder)

    if response.status_code == 200:
        bases = []
        for obj in response.json():
            for obj_type in obj.get("event-types", []):
                if obj_type.get("event-type") == label:
                    bases.append(obj_type.get("base-uri"))
                    break

        with open(
            f"{folder}{name} esmond data {source.split('-')[1]}-{destination.split('-')[1]} {today.strftime('%m-%d-%Y')}.csv",
            "w",
        ) as f:
            f.write("Timestamp,Data,Atraso(ms)\n")
            for link in bases:
                values = get_data(base + link + limite, time_range)
                for value in values:
                    f.write(
                        f"{value['ts']},{datetime.fromtimestamp(int(value['ts'])).strftime('%Y-%m-%d %H:%M:%S')},{calc_mean(value['val'])}\n"
                    )


def has_summary_3600(base_url, test_type, source, destination, label):
    """Verifica se há resumo com janela de 3600s para a métrica especificada"""
    header = {
        "pscheduler-test-type": test_type,
        "source": source,
        "destination": destination,
    }
    response = requests.get(base_url, params=header, verify=False)
    if response.status_code != 200:
        return False

    for obj in response.json():
        for obj_type in obj.get("event-types", []):
            if obj_type.get("event-type") == label:
                for summary in obj_type.get("summaries", []):
                    if summary.get("summary-type") == "aggregation" and summary.get("summary-window") == "3600":
                        return True
    return False



# def request_loss(folder, name, source, destination, type, time_range, label="packet-loss-rate-bidir"):
#     disable_warnings(InsecureRequestWarning)
#     url = "http://monipe-central.rnp.br/esmond/perfsonar/archive/?"
#     header = {
#         "pscheduler-test-type": type,
#         "source": source,
#         "destination": destination,
#         "time-range": time_range,
#     }

#     response = requests.get(url, params=header, verify=False)
#     print(f"URL: {response.url}")
#     print(f"Status Code: {response.status_code}")

#     try:
#         response_json = response.json()
#         print(f"Resposta: {response_json}")
#     except Exception as e:
#         print("Erro ao converter resposta para JSON:", e)
#         return

#     if not os.path.exists(folder):
#         os.makedirs(folder)

#     if response.status_code == 200:
#         bases = []
#         for obj in response_json:
#             for obj_type in obj.get("event-types", []):
#                 if obj_type.get("event-type") == label:
#                     found_summary = False
#                     for summary in obj_type.get("summaries", []):
#                         if summary.get("summary-type") == "aggregation" and summary.get("summary-window") == "3600":
#                             bases.append(summary.get("uri"))
#                             found_summary = True
#                             break
#                     if not found_summary:
#                         bases.append(obj_type.get("base-uri"))
#                     break

#         with open(
#             f"{folder}{name} esmond data {source.split('-')[1]}-{destination.split('-')[1]} {today.strftime('%m-%d-%Y')}.csv",
#             "w",
#         ) as f:
#             f.write("Timestamp,Data,Loss\n")
#             for link in bases:
#                 values = get_data(base + link + "?limit=285000", time_range)
#                 print(f"Base {link} retornou {len(values)} valores.")
#                 for value in values:
#                     ts = value.get("ts")
#                     val = value.get("val")

#                     if ts is None or val is None:
#                         print(f"Valor inválido ou ausente no timestamp {ts}")
#                         continue

#                     # Trata se for lista ou valor único
#                     if isinstance(val, list) and val:
#                         mean_val = calc_mean(val)
#                     elif isinstance(val, (float, int)):
#                         mean_val = round(float(val), 6)
#                     else:
#                         print(f"Valor inesperado no timestamp {ts}: {val}")
#                         continue

#                     dt_str = datetime.fromtimestamp(int(ts)).strftime('%Y-%m-%d %H:%M:%S')
#                     f.write(f"{ts},{dt_str},{mean_val}\n")

def get_data(link, time_range):
    header = {"time-range": time_range}
    response = requests.get(link, params=header, verify=False)
    try:
        return response.json()
    except:
        return []


def request_loss(folder, name, source, destination, type, time_range, label="packet-loss-rate-bidir"):
    disable_warnings(InsecureRequestWarning)

    url = f"{base}/esmond/perfsonar/archive/?"
    header = {
        "pscheduler-test-type": type,
        "source": source,
        "destination": destination,
        "time-range": time_range,
    }

    response = requests.get(url, params=header, verify=False)
    print(f"URL: {response.url}")
    print(f"Status Code: {response.status_code}")

    try:
        response_json = response.json()
    except Exception as e:
        print("Erro ao converter resposta para JSON:", e)
        return

    if not os.path.exists(folder):
        os.makedirs(folder)

    if response.status_code == 200:
        bases = []
        for obj in response_json:
            for obj_type in obj.get("event-types", []):
                if obj_type.get("event-type") == label:
                    # Apenas base-uri (sem agregação)
                    uri = obj_type.get("base-uri")
                    if uri:
                        bases.append(uri)
                    break

        filepath = f"{folder}{name} esmond data {source.split('-')[1]}-{destination.split('-')[1]} {today.strftime('%m-%d-%Y')}.csv"
        with open(filepath, "w") as f:
            f.write("Timestamp,Data,Loss\n")
            for link in bases:
                values = get_data(base + link + "?limit=285000", time_range)
                print(f"Base {link} retornou {len(values)} valores.")
                for value in values:
                    ts = value.get("ts")
                    val = value.get("val")

                    if ts is None or val is None:
                        print(f"Valor inválido ou ausente no timestamp {ts}")
                        continue

                    dt_str = datetime.fromtimestamp(int(ts)).strftime('%Y-%m-%d %H:%M:%S')

                    if isinstance(val, list):
                        for v in val:
                            f.write(f"{ts},{dt_str},{v}\n")
                    else:
                        f.write(f"{ts},{dt_str},{val}\n")


address = [
    "monipe-ce-banda.rnp.br",
    "monipe-ac-banda.rnp.br",
    "monipe-am-banda.rnp.br",
    "monipe-ap-banda.rnp.br",
    "monipe-ba-banda.rnp.br",
    "monipe-df-banda.rnp.br",
    "monipe-es-banda.rnp.br",
    "monipe-go-banda.rnp.br",
    "monipe-ma-banda.rnp.br",
    "monipe-mg-banda.rnp.br",
    "monipe-ms-banda.rnp.br",
    "monipe-mt-banda.rnp.br",
    "monipe-pa-banda.rnp.br",
    "monipe-pb-banda.rnp.br",
    "monipe-pe-banda.rnp.br",
    "monipe-pi-banda.rnp.br",
    "monipe-pr-banda.rnp.br",
    "monipe-rj-banda.rnp.br",
    "monipe-rn-banda.rnp.br",
    "monipe-ro-banda.rnp.br",
    "monipe-rr-banda.rnp.br",
    "monipe-rs-banda.rnp.br",
    "monipe-sc-banda.rnp.br",
    "monipe-se-banda.rnp.br",
    "monipe-sp-banda.rnp.br",
    "monipe-to-banda.rnp.br"
]
address_atraso = [
    "monipe-ce-atraso.rnp.br",
    "monipe-ac-atraso.rnp.br",
    "monipe-am-atraso.rnp.br",
    "monipe-ap-atraso.rnp.br",
    "monipe-ba-atraso.rnp.br",
    "monipe-df-atraso.rnp.br",
    "monipe-es-atraso.rnp.br",
    "monipe-go-atraso.rnp.br",
    "monipe-ma-atraso.rnp.br",
    "monipe-mg-atraso.rnp.br",
    "monipe-ms-atraso.rnp.br",
    "monipe-mt-atraso.rnp.br",
    "monipe-pa-atraso.rnp.br",
    "monipe-pb-atraso.rnp.br",
    "monipe-pe-atraso.rnp.br",
    "monipe-pi-atraso.rnp.br",
    "monipe-pr-atraso.rnp.br",
    "monipe-rj-atraso.rnp.br",
    "monipe-rn-atraso.rnp.br",
    "monipe-ro-atraso.rnp.br",
    "monipe-rr-atraso.rnp.br",
    "monipe-rs-atraso.rnp.br",
    "monipe-sc-atraso.rnp.br",
    "monipe-se-atraso.rnp.br",
    "monipe-sp-atraso.rnp.br",
    "monipe-to-atraso.rnp.br"
]
for source, source_atraso in zip(address, address_atraso):
    for destination, destination_atraso in zip(address, address_atraso):
        if source != destination and destination.split("-")[1] in ["rs"]:
            print(f"Origem: {source} → Destino: {destination}")

            # request_traceroute("datasets traceroute/", "traceroute", source_atraso, destination_atraso, "trace", "63072000")
            # request("datasets vazao/cubic/", "cubic", source, destination, "throughput", "63072000")
            # request("datasets vazao/bbr/", "bbr", source, destination, "throughput", "63072000", "10000000000")
            request_atraso("datasets atraso/", "atraso", source_atraso, destination_atraso, "latencybg", "63072000", "histogram-owdelay")
            # request_loss("datasets perda/", "loss_unidir", source_atraso, destination_atraso, "rtt", "63072000", label="packet-loss-rate-bidir")

Origem: monipe-ce-banda.rnp.br → Destino: monipe-rs-banda.rnp.br
URL: https://monipe-central.rnp.br/esmond/perfsonar/archive/?pscheduler-test-type=latencybg&source=monipe-ce-atraso.rnp.br&destination=monipe-rs-atraso.rnp.br&time-range=63072000
Status Code: 200
Resposta: [{'url': 'https://monipe-central.rnp.br/esmond/perfsonar/archive/287bb4d1c7504a76b80094f9337ab5ee/', 'metadata-key': '287bb4d1c7504a76b80094f9337ab5ee', 'subject-type': 'point-to-point', 'event-types': [{'base-uri': '/esmond/perfsonar/archive/287bb4d1c7504a76b80094f9337ab5ee/failures/base', 'event-type': 'failures', 'time-updated': None, 'summaries': []}, {'base-uri': '/esmond/perfsonar/archive/287bb4d1c7504a76b80094f9337ab5ee/histogram-owdelay/base', 'event-type': 'histogram-owdelay', 'time-updated': 1739611000, 'summaries': [{'uri': '/esmond/perfsonar/archive/287bb4d1c7504a76b80094f9337ab5ee/histogram-owdelay/aggregations/300', 'summary-type': 'aggregation', 'summary-window': '300', 'time-updated': 1739611000}, {'uri'

ConnectTimeout: HTTPConnectionPool(host='monipe-central.rnp.br', port=80): Max retries exceeded with url: /esmond/perfsonar/archive/?pscheduler-test-type=latencybg&source=monipe-es-atraso.rnp.br&destination=monipe-rs-atraso.rnp.br&time-range=63072000 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x0000020397B438A0>, 'Connection to monipe-central.rnp.br timed out. (connect timeout=None)'))